In [ ]:
import holoviews as hv
import numpy as np
import os 
from itertools import product
from cell_registration import CellReg
hv.extension('bokeh')

### Get animal data + check images

In [ ]:
ani='astro6'
idx = [0,1]
astro = CellReg(ani,'FOV1',3,session_inds=idx)
N = astro.N_sessions
images_re =astro.resize_images(image_type='max dff',shifted=False,session_inds=idx)
foots_re = astro.resize_foots()

#### Plot images from sessions
to display session lables pass image_titles=True

In [ ]:
astro.plot_im_stack(images_re,scale_factor=2)


### Choose session and align to find shift

In [ ]:
im1,im2=images_re[0],images_re[1]
astro.plot_overlaid_rgb(im1,im2,gain=5,alpha=.8)

#### Define the range of shifts to display

In [ ]:
rotations = np.linspace(-np.pi/100,np.pi/100,20)
translation_x = np.arange(-10,10,1)
translation_y = np.arange(-10,10,1)
translations= list(product(translation_x,translation_y))


In [ ]:
im_dict = {(translation[0],translation[1]): astro.plot_translate(im1,im2,shift_x=translation[0],shift_y=translation[1]) for translation in translations}
hmap = hv.HoloMap(im_dict, kdims=['translation_x', 'translation_y'])
hmap

##### Choose the translation values you want

In [ ]:
X = -3
Y = -1

##### Optional: Plot rotations

In [ ]:
rotations_1 = np.linspace(0,np.pi/100,10)
rotations_2 = np.linspace(-np.pi/100,0,10)
im_dict_rot_1 = {(rotation): astro.plot_translate(im1,im2,shift_x=X,shift_y=Y,rotation=rotation) for rotation in rotations_1}
im_dict_rot_2 = {(rotation): astro.plot_translate(im1,im2,shift_x=X,shift_y=Y,rotation=rotation) for rotation in rotations_2}

hmap_1 = hv.HoloMap(im_dict_rot_1, kdims=['rotation'])
hmap_2 = hv.HoloMap(im_dict_rot_2,kdims =['rotation'])
hmap_1

### Update shifts for all sessions

In [ ]:
choose_shiftX = [0,-3]
choose_shiftY = [0,-1]
choose_rotation = [0,0]
choose_shears = [0,0]

### Apply shifts to footprints and save footprints, images

In [ ]:
astro.export_affine_shift_images(images_re,choose_shiftX,choose_shiftY,choose_rotation,choose_shears)
astro.export_affine_shift_footprints(foots_re,choose_shiftX,choose_shiftY,choose_rotation,choose_shears,session_inds=idx)

In [ ]:
translations = {'shifts_X':choose_shiftX,'shifts_Y':choose_shiftY,'rotations':choose_rotation,'shears':choose_shears}
np.save(os.path.join(astro.base_directory,'CellReg',astro.animal+'_'+astro.FOV,'shifted_footprints','affine_transform'),translations)

### Fix footprint export bug - 

In [ ]:
# for ani in ['astro3','astro5','astro7','astro8','astro9']:
#     astro = CellReg(ani,'FOV1',5)
#     N = astro.N_sessions
#     images_re =astro.resize_images(image_type='max dff')
#     foots_re = astro.resize_foots()
    
#     translations = np.load(os.path.join(r"C:\Users\RamirezLab\Desktop\Rebecca\CellReg",astro.animal +'_'+astro.FOV,"shifted_footprints","affine_transform.npy"),allow_pickle=True).item()

#     choose_shiftX = translations['shifts_X']
#     choose_shiftY = translations['shifts_Y']
#     choose_rotation = translations['rotations']
#     choose_shears = translations['shears']

#     astro.export_affine_shift_footprints(foots_re,choose_shiftX,choose_shiftY,choose_rotation,choose_shears)

In [ ]:
# fixed error in output of affine transform
# for ani in ['astro3','astro5','astro7']:
#     file = os.path.join(r"C:\Users\RamirezLab\Desktop\Rebecca\CellReg",ani+"_FOV1","shifted_footprints","affine_transform.npy")
#     translations = np.load(file,allow_pickle=True).item()
#     translations['shifts_Y']=translations['shift_Y']
#     del translations['shift_Y']
#     np.save(os.path.join(r"C:\Users\RamirezLab\Desktop\Rebecca\CellReg",ani+"_FOV1","shifted_footprints","affine_transform.npy"),translations)

#     load = np.load(os.path.join(r"C:\Users\RamirezLab\Desktop\Rebecca\CellReg",ani+"_FOV1","shifted_footprints","affine_transform_.npy"),allow_pickle=True)
#     print(load)